# Chapter 2 — Calculating and Exploring Spectral Indices

This notebook calculates four vegetation and pigment-sensitive indices — **NDVI, CIRE, CRI2 and ARI2** — and explores them with index maps, histograms, an interactive index-to-index scatter plot, and a correlation matrix.

**Before starting:** keep this notebook beside `workshop_utils.py`, start with a fresh kernel and run the cells from top to bottom. The imagery is read straight from the workshop STAC catalogue, so there is nothing to download first.

**Memory note:** index values, statistics, histograms and correlations use every valid pixel. The scatter plot draws a random sample of pixels so the browser stays responsive.

## 1. Load libraries and shared workshop functions

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from IPython.display import display

from workshop_utils import (
    INDEX_INFO,
    WORKSHOP_INDICES,
    load_data,
    compute_indices,
    plot_indices,
    save_indices_to_tiff,
    interactive_raster_explorer,
)

pio.renderers.default = "plotly_mimetype+notebook_connected"
OUTPUT_DIR = Path("outputs")

## 2. Index catalogue

| Index | Formula | Main interpretation |
|---|---|---|
| NDVI | (NIR-842 − Red-668) / (NIR-842 + Red-668) | General vegetation density and greenness |
| CIRE | (NIR-842 / Red-Edge-705) − 1 | Total chlorophyll via red-edge reflectance |
| CRI2 | (1 / Green-531) − (1 / Red-Edge-705) | Carotenoids, often elevated under stress or senescence |
| ARI2 | NIR-842 × [(1 / Green-560) − (1 / Red-Edge-705)] | Anthocyanin index weighted by NIR for canopy variation |

## 3. Load data and calculate indices

`load_data` reads the study area from the workshop STAC catalogue into memory (this takes 10–20 seconds). `compute_indices` then adds the four indices to the dataset as new float32 variables. Pixels whose input reflectance is implausible (at or below 0.01, or above 1) are left as NaN in the indices, so they are ignored by every statistic and plot below.

In [ ]:
ds = compute_indices(load_data())
print(f"Image: {ds.sizes['y']:,} rows × {ds.sizes['x']:,} columns at {ds.attrs['pixel_size_m'] * 100:.1f} cm per pixel")
ds

## 4. Index formulas and scene statistics

Each index becomes one column of a pandas table (one row per pixel). `describe()` then summarises every valid pixel, skipping NaN.

In [ ]:
index_table = ds[WORKSHOP_INDICES].to_dataframe()

index_stats = index_table.describe(percentiles=[0.02, 0.5, 0.98]).T.astype({"count": int})
index_stats.insert(0, "Formula", [INDEX_INFO[name]["formula"] for name in WORKSHOP_INDICES])
index_stats.round(4)

## 5. Index maps

### 5.1 A single index

`plot_indices` returns a Plotly figure, so you can restyle it before showing it.

In [ ]:
fig = plot_indices(ds, indices=["NDVI"], cols=1, show=False)
fig.update_xaxes(title_font=dict(size=16), tickfont=dict(size=14))
fig.update_yaxes(title_font=dict(size=16), tickfont=dict(size=14))
fig.show(config={"scrollZoom": True, "displaylogo": False})

### 5.2 All four indices

`robust=True` (the default) uses each index's P2–P98 range for the colour scale, so a few extreme pixels do not wash out the map. Add `save_path=OUTPUT_DIR / "workshop_indices.html"` to also save the figure as a stand-alone HTML file.

In [ ]:
plot_indices(ds);

## 6. Save the indices as GeoTIFFs

Exports are full resolution and keep the source CRS and geotransform. The first call writes one multi-band file with all four indices; the second writes a separate file for each index you choose.

In [ ]:
save_indices_to_tiff(ds, out_path=OUTPUT_DIR / "workshop_indices.tif")
save_indices_to_tiff(ds, indices=["NDVI", "CIRE", "ARI2"], out_dir=OUTPUT_DIR / "indices");

## 7. Interactive index explorer

Choose an index to redraw its map and histogram. The histogram and statistic lines use every valid pixel.

### 7.1 Full data range (min, median, max)

In [ ]:
interactive_raster_explorer(
    ds,
    WORKSHOP_INDICES,
    metadata=INDEX_INFO,
    dropdown_description="Index:",
    robust=False,
    show_colorbar=True,
);

### 7.2 Robust colour stretch with index descriptions

Compare this view with 7.1: the P2–P98 stretch reveals spatial detail that a full-range colour scale hides when a few pixels are extreme.

In [ ]:
interactive_raster_explorer(
    ds,
    WORKSHOP_INDICES,
    metadata=INDEX_INFO,
    dropdown_description="Index:",
    bins=90,
    statistics=("mean", "p2", "p98"),
    show_colorbar=True,
    show_metadata=True,
);

## 8. Interactive index-to-index scatter plot

Drawing every pixel (millions of points) would freeze the browser, so the plot shows a random sample. Use the slider to change the sample size; the correlation coefficient in the title uses **all** valid pixel pairs.

In [ ]:
rng = np.random.default_rng(42)


def render_scatter(x_name, y_name, n_samples):
    pairs = index_table[[x_name, y_name]].dropna().to_numpy()
    r = np.corrcoef(pairs, rowvar=False)[0, 1]
    n = min(n_samples, len(pairs))
    sample = pairs[rng.choice(len(pairs), size=n, replace=False)]

    fig = go.Figure(go.Scattergl(
        x=sample[:, 0], y=sample[:, 1], mode="markers",
        marker=dict(size=3, opacity=0.35, color="#2c7fb8"),
        hovertemplate=f"{x_name}: %{{x:.4f}}<br>{y_name}: %{{y:.4f}}<extra></extra>",
    ))
    fig.update_layout(
        title=(f"{x_name} vs {y_name} — {n:,} sampled of {len(pairs):,} valid pixels "
               f"(Pearson r = {r:.3f}, all pixels)"),
        xaxis_title=x_name, yaxis_title=y_name,
        template="plotly_white", height=520,
        margin=dict(l=70, r=30, t=70, b=60),
    )
    fig.show(config={"displaylogo": False})


sc_x = widgets.Dropdown(options=WORKSHOP_INDICES, value="NDVI", description="X:")
sc_y = widgets.Dropdown(options=WORKSHOP_INDICES, value="CIRE", description="Y:")
sc_n = widgets.IntSlider(
    value=20_000, min=5_000, max=100_000, step=5_000,
    description="Samples:", continuous_update=False,
)
scatter_output = widgets.interactive_output(
    render_scatter, {"x_name": sc_x, "y_name": sc_y, "n_samples": sc_n}
)
display(widgets.VBox([widgets.HBox([sc_x, sc_y]), sc_n, scatter_output]))

## 9. Correlation between indices

Pearson correlations use every pixel that is valid in all four indices. The matrix is a small 4 × 4 table, so it is cheap to display.

In [ ]:
complete_pixels = index_table.dropna()
corr_matrix = complete_pixels.corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdYlGn",
    zmin=-1,
    zmax=1,
)
fig.update_layout(
    title=f"Pearson correlation matrix — n={len(complete_pixels):,} complete pixels",
    height=480,
    width=560,
    template="plotly_white",
)
fig.show(config={"displaylogo": False})

### 9.1 Correlation matrix with distributions

The lower triangle shows the correlations; the diagonal shows each index's smoothed distribution. The KDE uses a random sample of 50,000 pixels for speed — this affects only the drawn curve, not the correlations.

In [ ]:
values = corr_matrix.to_numpy()
n = len(WORKSHOP_INDICES)
upper = np.triu(np.ones_like(values, dtype=bool))  # hide the diagonal and upper triangle
corr_cmap = LinearSegmentedColormap.from_list("correlation", ["#5b8fa8", "#f2f2f2", "#c95735"])

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.ma.array(values, mask=upper), cmap=corr_cmap, vmin=-1, vmax=1)
ax.set_xticks(range(n), WORKSHOP_INDICES, rotation=90)
ax.set_yticks(range(n), WORKSHOP_INDICES)
for i in range(n):
    for j in range(i):
        ax.text(j, i, f"{values[i, j]:.2f}", ha="center", va="center", fontsize=10)

# Draw each index's distribution as a small KDE curve on the diagonal.
kde_sample = complete_pixels.sample(n=min(50_000, len(complete_pixels)), random_state=42)
for i, name in enumerate(WORKSHOP_INDICES):
    data = kde_sample[name].to_numpy()
    low, high = np.percentile(data, [1, 99])
    x_grid = np.linspace(low, high, 200)
    density = gaussian_kde(data[(data >= low) & (data <= high)])(x_grid)

    kde_ax = ax.inset_axes([i - 0.45, i - 0.45, 0.9, 0.9], transform=ax.transData)
    kde_ax.fill_between(x_grid, density, color="#5b8fa8", alpha=0.35)
    kde_ax.plot(x_grid, density, color="#5b8fa8", linewidth=1.2)
    kde_ax.axis("off")

ax.set_xticks(np.arange(-0.5, n, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1)
ax.tick_params(which="minor", bottom=False, left=False)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()

## Activities

1. Compare NDVI and CIRE. Can you find dense-canopy areas where CIRE retains variation but NDVI appears saturated?
2. Locate the highest-ARI2 cluster, then compare the same location in NDVI.
3. Identify the strongest positive and the weakest/most negative pairs in the correlation matrix. Which pigment proxy is least related to NDVI?

ARI2 and CRI2 can be negative where the reciprocal-difference formulas are applied to soil or shadow pixels. The scatter plot samples pixels to keep the browser responsive; its correlation value uses all pixels.